In [21]:
# Run with a Sagemath Kernel

In [22]:
reset()

In [23]:
%run ntru.ipynb
import subprocess, re
import numpy as np
from pathlib import Path

In [24]:
flatter_path = 'flatter' ## Don't forget to add your flatter path

In [25]:
def ntruhps(x):
    '''
    Returns the parameters of NTRU N,q,d, as well as our choice of N1,N2 and k
    :param x: The NTRU variant, 1 for ntruhps2048509, 2 for ntruhps2048677, 3 for ntruhps4096821
    '''
    if x == 1: # ntruhps2048509
        N = 509; q = 2048; d = q/16-1; k = 425; N1 = 9; N2 = 8
    elif x == 2: # ntruhps2048677
        N = 677; q = 2048; d = q/16-1; k = 600; N1 = 1; N2 = 15
    elif x == 3: # ntruhps4096821
        N = 821; q = 4096; d = q/16-1; k = 740; N1 = 7; N2 = 21
    else:
        return None
    return N, q, d, k, N1, N2

def centered_mod(arr, q):
    arr = np.array(arr)
    return ((arr + q // 2) % q) - q // 2

def calculate_c_minus_m(ciphertext, known_msg):
    '''
    :param ciphertext: the ciphertext
    :param known_msg: the known coefficients of the message
    :return: a list of the differences 3^-1(c-m)%q
    '''
    output = []
    for i in range(len(known_msg)): # we know only the first 3 coefficients of the message
        three_inverse = mod(3^(-1), q)
        output.append(int(three_inverse*(ciphertext[i] - known_msg[i])))
    return output

def get_coeffs_of_right_side(N, h, k):
    return [ h[(k-i)%N] for i in range(N) ]

def create_fplll_file(filename, n, a, a0, N1, N2, q, k):
    # Create the Bk matrix
    equations = len(a0)

    zerom = matrix(1, n, 0)
    M = identity_matrix(ZZ, n)
    M = M.stack(zerom)
    lowerM = [vector(a[i])*N2 for i in range(equations)]

    for i in range(equations):
        M = M.stack(lowerM[i])

    lastCol = [0]*n + [N1]
    for i in range(equations):
        lastCol.append(-N2*a0[i])
    lactCol = vector(QQ, lastCol)
    lactCol = lactCol.column()
    M = M.augment(lactCol)
    Mt = M.transpose() # write to file and read from fpylll

    qIN = identity_matrix(ZZ, k)*q*N2
    zerom2 = matrix(k, n+1, 0)
    qIN = zerom2.augment(qIN)
    E = Mt.stack(qIN)

    f = open("matrices/"+filename, 'w')
    f.write("[")
    for i in E:
        f.write("[")
        [f.write(str(j)+" ") for j in list(i)]
        f.write("]\n")
    f.write("]\n")
    f.close()
    return E

def update_basis(Mt, known_r, N2, k1):
    '''
    Finds the elements from known_r vector that have value of zero, one, minus one and updates basis Mt by deleting its corresponding columns
    :param Mt: Our basis matrix
    :return: the positions of the zeros, ones, minus ones in the known_r vector the updated a and a0 vectors
    '''
    temp = Mt[0:N,N+1:N+k1+1] # get the last k elements of the first N rows
    sys_matrix = 1/N2 * matrix(temp).T # get the original system matrix by dividing temp with N2

    zero_pos       = [i for i, x in enumerate(known_r) if x ==  0] # the position of zeros in r at the first k-entries
    one_pos       = [i for i, x in enumerate(known_r) if x ==  1] # the position of ones in r at the first k-entries
    minus_one_pos = [i for i, x in enumerate(known_r) if x == -1] # the position of minus ones in r at the first k-entries
    positions = sorted(set(zero_pos + one_pos + minus_one_pos))

    # Safe vector-zero for sums
    zero = vector([0] * sys_matrix.nrows())

    # Column sums for 1 and -1 positions
    Sum       = sum((vector(sys_matrix.column(i)) for i in one_pos),       zero)
    Sum_minus = sum((vector(sys_matrix.column(i)) for i in minus_one_pos), zero)

    # Delete columns at known positions
    sys_matrix_del = sys_matrix.delete_columns(positions)

    # Update RHS a0: subtract ones, add minus-ones
    s0      = vector(a0) - Sum + Sum_minus
    new_a0  = list(s0 % q)
    new_a   = [list(row) for row in sys_matrix_del.rows()]
    return positions, zero_pos, one_pos, minus_one_pos, new_a, new_a0

def positions_of_N1(lst, N1):
    '''
    Given a list lst, we provide the positions where a multiple of N1 occurs.
    eg. for N1=10, and lst = [10,0,0,10] then we get [0,3]
    :return: a list of the positions of N1 in lst
    '''
    if N1 == 0:
        raise ValueError("N1 must be non-zero.")
    positions = []
    for i, x in enumerate(lst):
        if x != 0 and ((x==N1) or (x==-N1)):
            positions.append(i)
    return positions

def set_zeros_ones_and_minus_ones(q_del, zero_pos, one_pos, minus_one_pos):
    """
    Reconstruct the original list given:
    :param q_del: list after deleting entries at positions zero_pos ∪ one_pos ∪ minus_one_pos
    :param zero_pos: positions where 0s were deleted
    :param one_pos: positions where 1s were deleted
    :param minus_one_pos: positions where -1s were deleted
    :return res the list with the ones, zeros and minus ones added again
    """
    s0, s1, sm1 = set(zero_pos), set(one_pos), set(minus_one_pos)

    # Overlap check
    if (s0 & s1) or (s0 & sm1) or (s1 & sm1):
        raise ValueError("Index sets overlap")

    n_total = len(q_del) + len(s0) + len(s1) + len(sm1)

    # Precompute total deleted positions
    deleted = s0 | s1 | sm1

    # Consistency check
    expected_qdel_len = n_total - len(deleted)
    if len(q_del) != expected_qdel_len:
        raise ValueError(
            f"q_del length ({len(q_del)}) != n_total - |deleted| "
            f"({n_total} - {len(deleted)} = {expected_qdel_len})"
        )

    res = []
    it = iter(q_del)
    for i in range(n_total):
        if i in s0:
            res.append(0)
        elif i in s1:
            res.append(1)
        elif i in sm1:
            res.append(-1)
        else:
            res.append(next(it))
    return res

def check_result(result):
    norm_r = vector(r[0]).norm().n()
    norm_result = vector(result).norm().n()
    # print("Norms of r and result: ", norm_r, norm_result)
    # if approx_value[variant] -3 <= norm_result <= approx_value[variant] + 3:
    if norm_r == norm_result:
        if r[0] == list(result):
            show("Correct r found: ", r[0] == list(result))
            return True
        else:
            return False
    else:
        return False

def try_row(possible_solution, homoge):
    if check_result(possible_solution):
        return True
    result_minus_one = list(vector(possible_solution)-vector([1]*N)) # we use the vector from the reduction minus a [1]*N vector
    if check_result(result_minus_one):
        return True
    result_plus_one = list(vector(possible_solution)+vector([1]*N)) # we use the vector from the reduction plus a [1]*N vector
    if check_result(result_plus_one):
        return True
    result_center =  centered_mod(possible_solution, q) # centerlift the reduction vector
    if check_result(result_center):
        return True
    result_minus = centered_mod( list(vector(possible_solution)-vector([1]*N)) , q) # centerlifted vector minus [1]*N vector
    if check_result(result_minus):
        return True
    result_plus = centered_mod( list(vector(possible_solution)+vector([1]*N)) , q) # centerlifted vector plus [1]*N vector
    if check_result(result_plus):
        return True
    result_minus_center = centered_mod( list(vector(result_center)-vector([1]*N)) , q) # centerlifted vector minus [1]*N vector
    if check_result(result_minus_center):
        return True
    result_plus_center = centered_mod( list(vector(result_center)+vector([1]*N)) , q) # centerlifted vector plus [1]*N vector
    if check_result(result_plus_center):
        return True

    for i in homoge:
        plus = possible_solution + i # add the homogeneous vector to our possible solution
        minus = possible_solution - i # subtract the homogeneous vector to our possible solution
        if check_result(plus):
            return True
        if check_result(minus):
            return True
    return False

def create_NTRU(N, q, d, k):
    Zx.<x> = ZZ[]
    D = x^N - 1

    R.<xN> = Zx.quotient(D) # ring in Zx/DN

    F2 = GF(2)
    F2x.<x2> = F2[]
    S2.<X2> = F2x.quotient(D) # ring in 2/DN

    F3 = GF(3)
    F3x.<x3> = F3[]
    S3.<X3> = F3x.quotient(D) # ring in 3/DN

    Sq.<x> = PolynomialRing(Integers(q)) # ring modulo q

    # Create the NTRU polynomials
    while True:
        f, g, f3, fq, h = gen_keys(N, q, d, Zx, S3, Sq, S2, R)
        if fq is not None: # if an f invertible in q is found
            break
    msg = gen_msg(N, d, Zx)
    cipher, r = encrypt(N, d, msg[1], h, Zx, Sq, R)     # the ciphertext and the ephemeral key
    decr_m = decrypt(cipher, f, f3, msg[1], Zx, S3, Sq, R)    # we check if the decryption is correct

    # Compute vectors a and a0
    known_msg = msg[0][:k]
    a0 = calculate_c_minus_m(cipher, known_msg)
    a = []
    for i in range(k):
        a.append(get_coeffs_of_right_side(N, h, i))

    return msg, r, cipher, a, a0, known_msg

In [ ]:
variant = 1
experiments = 10
approx_value = {1:19, 2:21, 3:24} # 1 stands for ntruhps2048509, 2 for ntruhps2048677, 3 for ntruhps4096821
counter = 0
for experiment in range(experiments):
    print(experiment, "----------------------------------------------------------------------")
    found = False
    N,q,d,k,N1,N2 = ntruhps(variant)
    # N = 53; q = 2^10; d = 1; N1 = 10; N2 = 4; k=25
    # N = 11; q = 2^6; d = 4; N1 = 10; N2 = 3; k = 4

    k1 = 300
    k2 = 135

    print("k/N * 100:",(k1/N).n() * 100)
    print("(k1+k2)/(2*N) * 100:",((k1+k2)/(2*N)).n() * 100)
    msg, r, cipher, a, a0, known_msg = create_NTRU(N, q, d, k1)
    known_r = list(r[0])[:k2]

    for j in range(1): # change to range(-2,3) to test more N1 values
        tempN1 = N1 + j
        for l in range(1): # change to range(-2,3) to test more N2 values
            temp = N2 + l
            tempN2 = ceil(q^temp)
            our_set = {-1,0,1}
            print("N1, N2 = ", tempN1, temp)

            filename = str(N)+"_"+str(k1)
            Mt = create_fplll_file(filename, N, a, a0, tempN1, tempN2, q, k1)

            current_path = Path.cwd()
            matrix_path = str(current_path) + "/matrices/" + filename
            command = flatter_path + " < "+ matrix_path

            positions, zero_pos, one_pos, minus_one_pos, new_a, new_a0 = update_basis(Mt, known_r, tempN2, k1)
            new_N = N - len(positions)

            # create and write our updated basis in a file
            Mt_new = create_fplll_file(filename, new_N, new_a, new_a0, tempN1, tempN2, q, k1)
            print("Original basis dimension:", Mt.dimensions(),", updated dimension: ", Mt_new.dimensions(), ", difference: ", len(positions))

            output = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).stdout
            rows = output.strip("[] \n").split('\n')
            matrix_data = [list(map(int, re.findall(r'-?\d+', row))) for row in rows]
            A = matrix(matrix_data)

            output = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).stdout
            rows = output.strip("[] \n").split('\n')
            matrix_data = [list(map(int, re.findall(r'-?\d+', row))) for row in rows]
            A = matrix(matrix_data)

            target_column = [A[i][new_N] for i in range(new_N+k1+1)] # in this column we are looking for multiples of N1
            lst = positions_of_N1(target_column, tempN1)
            solutions = []
            for i in lst:
                result_incomplete = list(A[i][:new_N]) # this result vector doesn't have all its elements yet
                result = set_zeros_ones_and_minus_ones(result_incomplete, zero_pos, one_pos, minus_one_pos) # the full result vector
                result_minus_incomplete = [-x for x in result_incomplete] # this result vector doesn't have all its elements yet
                result_minus = set_zeros_ones_and_minus_ones(result_minus_incomplete, zero_pos, one_pos, minus_one_pos) # the full result vector
                is_solution    = np.all(np.matrix(a)@np.array(result)%q == np.array(a0)%q)
                is_solution_minus =  np.all(np.matrix(a)@np.array(result_minus)%q == np.array(a0)%q)
                # print(is_solution, is_solution_minus)
                solutions.append(result)
                solutions.append(result_minus)

            for solution in solutions:
                if try_row(solution, []):
                    found = True
                    counter += 1
                    break # no need to see the other solutions
            if found:
                break # break from N2 loop
        if found:
            break # break from N1 loop

print("=========================================================")
print("Success rate %: ", counter*100/experiments)

0 ----------------------------------------------------------------------
k/N * 100: 58.9390962671906
(k1+k2)/(2*N) * 100: 42.7308447937132
N1, N2 =  9 8
